# Chapter 4 — Results Production Notebook

This notebook orchestrates all figures and tables for the restructured Chapter 4.  
Each section mirrors the thesis structure. Run cells individually to iterate on specific figures.

**Structure:**
- §4.1 Corpus Overview and Register Distinctiveness
- §4.2 Lexical Patterns Across Registers (keyness, exclusivity, co-occurrence)
- §4.3 Thematic Structure: STM
- §4.4 Interpretive Synthesis: H1a–H1c with Close Reading

**Prerequisites:** All pipeline scripts (00–03b) must have been run. Database `scraping_2.db` must be populated.

In [1]:
# ── Imports and Setup ──────────────────────────────────────────────────────
import sqlite3
import json
import math
import textwrap
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

try:
    import seaborn as sns
    sns.set_style('whitegrid')
except ImportError:
    pass

try:
    import networkx as nx
    _HAS_NX = True
except ImportError:
    _HAS_NX = False
    print('WARNING: networkx not installed — topic network figure will be skipped')

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'serif'


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/yannickkayser/Uni/Master/Master_Thesis/DarkSideofAI/venv/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/yannickkayser/Uni/Master/Master_Thesi

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [2]:
import matplotlib
import seaborn

print(matplotlib.__version__)
print(seaborn.__version__)

AttributeError: module 'matplotlib.cm' has no attribute 'register_cmap'

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

DB_PATH = 'data/scraping_2.db'        # adjust if needed
FIG_DIR = Path('output/chapter4/')     # all Chapter 4 figures saved here
FIG_DIR.mkdir(parents=True, exist_ok=True)

DPI_PUB = 300
DPI_EXP = 150
FIG_FORMAT = 'pdf'   # 'pdf' for vector (thesis), 'png' for quick preview

# ── Colour Palette (consistent across all figures) ─────────────────────────
C_CLIENT  = '#1B4F8A'   # deep blue  — client (B2B)
C_WORKER  = '#C0392B'   # deep red   — worker (B2W)
C_SHARED  = '#6C757D'   # grey       — shared / neutral
C_H1C     = '#E67E22'   # orange     — H1c (strategic hypervisibility)
C_GRID    = '#DEE2E6'
C_TEXT    = '#1A1A2E'

PAL_HYP = {
    'H1a': C_WORKER,
    'H1b': C_CLIENT,
    'H1c': C_H1C,
    None:  C_SHARED,
}

# ── Database connection ────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

# Quick check: which tables exist?
tables = {r[0] for r in conn.execute(
    "SELECT name FROM sqlite_master WHERE type IN ('table','view')"
).fetchall()}
print('Available tables/views:', sorted(tables))

In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────

def save_fig(fig, name, fmt=FIG_FORMAT, dpi=DPI_PUB):
    """Save figure to FIG_DIR with consistent settings."""
    path = FIG_DIR / f'{name}.{fmt}'
    fig.savefig(str(path), dpi=dpi, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')

def apply_style(ax):
    """Apply consistent thesis style to an axis."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color(C_GRID)
    ax.spines['bottom'].set_color(C_GRID)
    ax.tick_params(colors=C_TEXT, labelsize=9)
    ax.set_axisbelow(True)

def to_latex_table(df, caption, label, note='', fmt_cols=None):
    """Convert a DataFrame to a thesis-ready LaTeX tabular string."""
    lines = []
    lines.append(r'\begin{table}[ht]')
    lines.append(r'  \centering')
    lines.append(f'  \\caption{{{caption}}}')
    lines.append(f'  \\label{{{label}}}')
    lines.append(r'  \small')
    col_fmt = 'l' + 'r' * (len(df.columns) - 1)
    lines.append(f'  \\begin{{tabular}}{{{col_fmt}}}')
    lines.append(r'    \toprule')
    headers = ' & '.join(str(c) for c in df.columns) + r' \\'
    lines.append(f'    {headers}')
    lines.append(r'    \midrule')
    for _, row in df.iterrows():
        vals = ' & '.join(str(v) for v in row.values) + r' \\'
        lines.append(f'    {vals}')
    lines.append(r'    \bottomrule')
    lines.append(r'  \end{tabular}')
    if note:
        lines.append(r'  \begin{tablenotes}')
        lines.append(r'    \small')
        lines.append(f'    \\item \\textit{{Note.}} {note}')
        lines.append(r'  \end{tablenotes}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

---
## §4.1 — Corpus Overview and Register Distinctiveness

In [ ]:
# ── §4.1.1: Corpus composition stats ──────────────────────────────────────
# These numbers go directly into the text of §4.1

stats = pd.read_sql("""
    SELECT
        audience,
        COUNT(DISTINCT domain) AS n_platforms,
        COUNT(*)               AS n_pages,
        SUM(token_count)       AS total_tokens
    FROM corpus_view
    WHERE page_id NOT IN (SELECT page_id FROM excluded_pages)
      AND audience != 'both'
    GROUP BY audience
""", conn)

total_pages = stats['n_pages'].sum()
stats['pct'] = (stats['n_pages'] / total_pages * 100).round(1)

print('=== Corpus Composition ===')
print(stats.to_string(index=False))
print(f'\nTotal pages (excl. both): {total_pages}')

In [ ]:
# ── §4.1.2: Lexical diversity (TTR) ───────────────────────────────────────

ttr = pd.read_sql("""
    SELECT
        audience,
        SUM(token_count) AS total_tokens,
        COUNT(*) AS n_pages
    FROM corpus_view
    WHERE page_id NOT IN (SELECT page_id FROM excluded_pages)
      AND audience != 'both'
    GROUP BY audience
""", conn)

# Unique tokens need to be computed from the actual unigram lists
for aud in ['worker', 'client']:
    rows = conn.execute("""
        SELECT unigrams FROM corpus_view
        WHERE audience = ? AND page_id NOT IN (SELECT page_id FROM excluded_pages)
    """, (aud,)).fetchall()
    all_terms = set()
    for r in rows:
        if r[0]:
            tokens = json.loads(r[0]) if isinstance(r[0], str) else r[0]
            if isinstance(tokens, list):
                all_terms.update(tokens)
            elif isinstance(tokens, dict):
                all_terms.update(tokens.keys())
    mask = ttr['audience'] == aud
    ttr.loc[mask, 'unique_tokens'] = len(all_terms)

ttr['ttr'] = ttr['unique_tokens'] / ttr['total_tokens']

print('=== Lexical Diversity ===')
print(ttr.to_string(index=False))

In [ ]:
# ── §4.1.3: Aggregate JSD and Cosine ──────────────────────────────────────
# These values were computed by 02_step1_analysis.py section E
# and may be stored in the DB or printed to stdout during that run.
# Check if they're in a table:

if 'aggregate_distance' in tables:
    agg = pd.read_sql('SELECT * FROM aggregate_distance', conn)
    print(agg)
elif 'distinctiveness_matrix' in tables:
    # Compute from the pairwise matrix
    print('aggregate_distance table not found.')
    print('Computing from distinctiveness_matrix...')
    # TODO: compute aggregate JSD between all-client vs all-worker vectors
    print('ACTION NEEDED: add aggregate computation here or extract from script logs')
else:
    print('Neither aggregate_distance nor distinctiveness_matrix found in DB.')
    print('Run 02_step1_analysis.py first.')

In [ ]:
# ── §4.1.4: JSD Platform MDS Map (Figure 4.1) ─────────────────────────────
# Reads pairwise JSD from distinctiveness_matrix, projects via MDS

from sklearn.manifold import MDS

if 'distinctiveness_matrix' not in tables:
    print('SKIP: distinctiveness_matrix not in DB. Run 02_step1_analysis.py.')
else:
    dm = pd.read_sql('SELECT * FROM distinctiveness_matrix', conn)
    
    # Pivot to square distance matrix
    # Expected columns: domain_a, domain_b, jsd, cosine_sim
    domains = sorted(set(dm['domain_a']) | set(dm['domain_b']))
    n = len(domains)
    d_idx = {d: i for i, d in enumerate(domains)}
    dist = np.zeros((n, n))
    for _, row in dm.iterrows():
        i, j = d_idx[row['domain_a']], d_idx[row['domain_b']]
        dist[i, j] = dist[j, i] = row['jsd']
    
    # MDS projection
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
    coords = mds.fit_transform(dist)
    
    # Get audience type per domain
    aud_map = dict(conn.execute("""
        SELECT DISTINCT domain, audience FROM corpus_view
        WHERE page_id NOT IN (SELECT page_id FROM excluded_pages)
    """).fetchall())
    
    # Within-company pairs
    PAIRS = [
        ('appen.com', 'crowdgen.com'),
        ('toloka.ai', 'mindrift.ai'),
        ('centific.com', 'oneforma.com'),
        ('labelbox.com', 'alignerr.com'),
        ('scale.com', 'remotasks.com'),
    ]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for i, d in enumerate(domains):
        aud = aud_map.get(d, 'unknown')
        color = C_CLIENT if aud == 'client' else C_WORKER if aud == 'worker' else C_SHARED
        marker = 'o' if aud == 'client' else '^' if aud == 'worker' else 's'
        ax.scatter(coords[i, 0], coords[i, 1], c=color, marker=marker,
                   s=60, zorder=3, edgecolors='white', linewidth=0.5)
        # Label
        ax.annotate(d.replace('.com','').replace('.ai','').replace('.org',''),
                    (coords[i, 0], coords[i, 1]),
                    fontsize=6, color=C_TEXT, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points')
    
    # Draw lines for within-company pairs
    for d1, d2 in PAIRS:
        if d1 in d_idx and d2 in d_idx:
            i1, i2 = d_idx[d1], d_idx[d2]
            ax.plot([coords[i1, 0], coords[i2, 0]],
                    [coords[i1, 1], coords[i2, 1]],
                    color=C_SHARED, linewidth=1, linestyle='--', alpha=0.6, zorder=1)
    
    # Legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=C_CLIENT,
                   markersize=8, label='Client-facing'),
        plt.Line2D([0], [0], marker='^', color='w', markerfacecolor=C_WORKER,
                   markersize=8, label='Worker-facing'),
        plt.Line2D([0], [0], color=C_SHARED, linestyle='--',
                   linewidth=1, label='Within-company pair'),
    ]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    apply_style(ax)
    ax.set_xlabel('MDS Dimension 1', fontsize=10)
    ax.set_ylabel('MDS Dimension 2', fontsize=10)
    ax.set_title('Platform Vocabulary Distance (Jensen-Shannon Divergence)', fontsize=12)
    
    save_fig(fig, 'fig_41_jsd_platform_map')
    plt.show()

---
## §4.2 — Lexical Patterns Across Registers

### §4.2.1 — Keyness Analysis

In [ ]:
# ── §4.2.1a: Top 20 Key Terms — Worker-facing ─────────────────────────────

worker_key = pd.read_sql("""
    SELECT term, g2, freq_worker_pmw, freq_client_pmw, exclusivity
    FROM keyness_results
    WHERE direction = 'worker'
    ORDER BY g2 DESC
    LIMIT 20
""", conn)

# Rename for display
worker_key.columns = ['Term', 'G²', 'Freq (B2W)', 'Freq (B2B)', 'E']
worker_key['G²'] = worker_key['G²'].round(1)
worker_key['Freq (B2W)'] = worker_key['Freq (B2W)'].round(1)
worker_key['Freq (B2B)'] = worker_key['Freq (B2B)'].round(1)
worker_key['E'] = worker_key['E'].round(3)

print('Top 20 Worker-Facing Key Terms')
print(worker_key.to_string(index=False))

# Export LaTeX
tex = to_latex_table(
    worker_key,
    caption='Top 20 key terms in the worker-facing sub-corpus ($G^2$, sorted descending). '
            'Freq = frequency per million tokens. $E$ = exclusivity index.',
    label='tab:worker-keyterms',
)
Path(FIG_DIR / 'tab_worker_keyterms.tex').write_text(tex)
print(f'\nLaTeX saved to {FIG_DIR}/tab_worker_keyterms.tex')

In [ ]:
# ── §4.2.1b: Top 20 Key Terms — Client-facing ─────────────────────────────

client_key = pd.read_sql("""
    SELECT term, g2, freq_client_pmw, freq_worker_pmw, exclusivity
    FROM keyness_results
    WHERE direction = 'client'
    ORDER BY g2 DESC
    LIMIT 20
""", conn)

client_key.columns = ['Term', 'G²', 'Freq (B2B)', 'Freq (B2W)', 'E']
client_key['G²'] = client_key['G²'].round(1)
client_key['Freq (B2B)'] = client_key['Freq (B2B)'].round(1)
client_key['Freq (B2W)'] = client_key['Freq (B2W)'].round(1)
client_key['E'] = client_key['E'].round(3)

print('Top 20 Client-Facing Key Terms')
print(client_key.to_string(index=False))

tex = to_latex_table(
    client_key,
    caption='Top 20 key terms in the client-facing sub-corpus ($G^2$, sorted descending).',
    label='tab:client-keyterms',
)
Path(FIG_DIR / 'tab_client_keyterms.tex').write_text(tex)
print(f'\nLaTeX saved to {FIG_DIR}/tab_client_keyterms.tex')

In [ ]:
# ── §4.2.1c: Keyness Diverging Bar Chart (Figure 4.2) ─────────────────────

top_n = 20

w = pd.read_sql(f"""
    SELECT term, g2 FROM keyness_results
    WHERE direction = 'worker' ORDER BY g2 DESC LIMIT {top_n}
""", conn)
c = pd.read_sql(f"""
    SELECT term, g2 FROM keyness_results
    WHERE direction = 'client' ORDER BY g2 DESC LIMIT {top_n}
""", conn)

# Worker terms get negative G² for the diverging chart
w['g2'] = -w['g2']

combined = pd.concat([w, c]).sort_values('g2')

fig, ax = plt.subplots(figsize=(10, 10))
colors = [C_WORKER if v < 0 else C_CLIENT for v in combined['g2']]
ax.barh(range(len(combined)), combined['g2'], color=colors, height=0.7)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(combined['term'], fontsize=9)
ax.axvline(0, color=C_TEXT, linewidth=0.8)
ax.set_xlabel('Log-likelihood G² (worker ← → client)', fontsize=10)
ax.set_title(f'Top {top_n} Key Terms per Register', fontsize=12)

legend_elements = [
    mpatches.Patch(facecolor=C_WORKER, label='Worker-facing'),
    mpatches.Patch(facecolor=C_CLIENT, label='Client-facing'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

apply_style(ax)
save_fig(fig, 'fig_42_keyness_diverging_bar')
plt.show()

### §4.2.2 — Term Exclusivity

In [ ]:
# ── §4.2.2: Exclusivity Scatter / Volcano (Figure 4.3) ────────────────────

if 'term_exclusivity' not in tables:
    print('SKIP: term_exclusivity not in DB.')
else:
    excl = pd.read_sql("""
        SELECT e.term, e.exclusivity_index, e.category,
               k.g2, k.direction
        FROM term_exclusivity e
        LEFT JOIN keyness_results k ON e.term = k.term
        WHERE k.g2 IS NOT NULL AND k.g2 > 10
    """, conn)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    colors = []
    for _, row in excl.iterrows():
        if row['exclusivity_index'] < -0.5:
            colors.append(C_WORKER)
        elif row['exclusivity_index'] > 0.5:
            colors.append(C_CLIENT)
        else:
            colors.append(C_SHARED)
    
    ax.scatter(excl['exclusivity_index'], excl['g2'],
               c=colors, s=15, alpha=0.5, zorder=2)
    
    # Label top terms
    top = excl.nlargest(15, 'g2')
    for _, row in top.iterrows():
        ax.annotate(row['term'], (row['exclusivity_index'], row['g2']),
                    fontsize=7, ha='center', va='bottom',
                    xytext=(0, 4), textcoords='offset points')
    
    ax.axvline(-0.5, color=C_WORKER, linewidth=0.8, linestyle=':', alpha=0.5)
    ax.axvline(0.5, color=C_CLIENT, linewidth=0.8, linestyle=':', alpha=0.5)
    ax.axvline(0, color=C_SHARED, linewidth=0.5, alpha=0.3)
    
    ax.set_xlabel('Exclusivity Index (← worker-exclusive | client-exclusive →)', fontsize=10)
    ax.set_ylabel('Keyness (G²)', fontsize=10)
    ax.set_title('Term Exclusivity vs Keyness', fontsize=12)
    
    apply_style(ax)
    save_fig(fig, 'fig_43_exclusivity_volcano')
    plt.show()
    
    # Stats for text
    n_worker_excl = len(excl[excl['exclusivity_index'] < -0.5])
    n_client_excl = len(excl[excl['exclusivity_index'] > 0.5])
    print(f'\nTerms with E < -0.5 (worker-exclusive): {n_worker_excl}')
    print(f'Terms with E > 0.5 (client-exclusive): {n_client_excl}')

### §4.2.3 — Co-occurrence Analysis

In [ ]:
# ── §4.2.3a: "Human" PMI Profile — B2B vs B2W (Figure 4.4) ─────────────

FOCUS_TERM = 'human'
COOC_N = 15

def get_pmi_profile(term, audience, top_n=COOC_N):
    """Get top PMI collocates for a term in a given audience."""
    rows = conn.execute("""
        SELECT collocate, pmi, co_freq
        FROM cooccurrence_results
        WHERE term = ? AND audience = ?
        ORDER BY pmi DESC
        LIMIT ?
    """, (term, audience, top_n)).fetchall()
    return pd.DataFrame(rows, columns=['collocate', 'pmi', 'co_freq'])

human_client = get_pmi_profile(FOCUS_TERM, 'client')
human_worker = get_pmi_profile(FOCUS_TERM, 'worker')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), sharey=False)

# Client side
if len(human_client) > 0:
    ax1.barh(range(len(human_client)), human_client['pmi'],
             color=C_CLIENT, height=0.7)
    ax1.set_yticks(range(len(human_client)))
    ax1.set_yticklabels(human_client['collocate'], fontsize=9)
    ax1.invert_yaxis()
ax1.set_title(f'"{FOCUS_TERM}" — Client-facing', fontsize=11)
ax1.set_xlabel('PMI', fontsize=10)
apply_style(ax1)

# Worker side
if len(human_worker) > 0:
    ax2.barh(range(len(human_worker)), human_worker['pmi'],
             color=C_WORKER, height=0.7)
    ax2.set_yticks(range(len(human_worker)))
    ax2.set_yticklabels(human_worker['collocate'], fontsize=9)
    ax2.invert_yaxis()
ax2.set_title(f'"{FOCUS_TERM}" — Worker-facing', fontsize=11)
ax2.set_xlabel('PMI', fontsize=10)
apply_style(ax2)

fig.suptitle(f'Co-occurrence Profiles of "{FOCUS_TERM}" Across Registers',
             fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig_44_cooc_human_comparison')
plt.show()

In [ ]:
# ── §4.2.3b: Automation Terms PMI Profiles (Figure 4.5) ───────────────────
# Repeat for automation seed terms

AUTO_TERMS = ['automate', 'ai', 'model', 'autonomous']
# Adjust this list based on which terms exist in your cooccurrence_results

for term in AUTO_TERMS:
    client_prof = get_pmi_profile(term, 'client', 10)
    worker_prof = get_pmi_profile(term, 'worker', 10)
    
    if len(client_prof) == 0 and len(worker_prof) == 0:
        print(f'  No co-occurrence data for "{term}" — skipping')
        continue
    
    print(f'\n=== "{term}" ===')
    print(f'Client top collocates: {", ".join(client_prof["collocate"].tolist()[:8])}')
    print(f'Worker top collocates: {", ".join(worker_prof["collocate"].tolist()[:8])}')

---
## §4.3 — Thematic Structure: Structural Topic Model

In [ ]:
# ── §4.3.0: Check STM tables exist ────────────────────────────────────────

stm_tables = {'stm_theta', 'stm_topic_terms', 'stm_prevalence'}
missing = stm_tables - tables
if missing:
    print(f'MISSING STM tables: {missing}')
    print('Run 03b_import_stm.py first.')
else:
    print('All STM tables present.')
    n_topics = conn.execute('SELECT COUNT(DISTINCT topic_id) FROM stm_topic_terms').fetchone()[0]
    print(f'Number of topics: {n_topics}')

In [ ]:
# ── §4.3.1: Full Topic Table (Table 4.X) ──────────────────────────────────
# The master table of the chapter: all 25 topics with FREX, estimate, CI

if 'stm_prevalence' in tables and 'stm_topic_terms' in tables:
    # Get prevalence estimates
    prev = pd.read_sql("""
        SELECT topic_id, frex_label, estimate, std_err,
               ci_lower, ci_upper, significant, direction
        FROM stm_prevalence
        ORDER BY estimate
    """, conn)
    
    # Get FREX terms per topic
    frex = pd.read_sql("""
        SELECT topic_id, GROUP_CONCAT(frex_term, ', ') AS frex_terms
        FROM (
            SELECT topic_id, frex_term
            FROM stm_topic_terms
            WHERE rank <= 7
            ORDER BY topic_id, rank
        )
        GROUP BY topic_id
    """, conn)
    
    topic_table = prev.merge(frex, on='topic_id', how='left')
    
    # Apply custom labels from 06_stm_results_figures.py
    TOPIC_LABELS = {
        1: 'Freelance AI training work',
        2: 'Survey & market research',
        3: 'NLP & text annotation',
        4: 'B2B outsourcing content',
        5: 'Technical AI evaluation tasks',
        6: 'AI governance & oversight',
        7: 'LLM evaluation & fine-tuning',
        8: 'Cookie & tracking notices',
        9: 'Computer vision labeling',
        10: 'Privacy policy & terms of service',
        11: 'Worker payments & earnings',
        12: 'Annotation platform tools',
        13: 'Government & compliance hiring',
        14: 'Expert specialist recruitment',
        15: 'Enterprise partnerships',
        16: 'B2B workforce solutions',
        17: 'Annotation API & developer tools',
        18: 'Human-AI discourse',
        19: 'Speech & audio data collection',
        20: 'Autonomous vehicles & drone sensing',
        21: 'AI deployment & synthetic data',
        22: 'Microwork platform campaigns',
        23: 'Medical image annotation',
        24: 'Search relevance & content tagging',
        25: 'Machine learning concepts',
    }
    topic_table['label'] = topic_table['topic_id'].map(TOPIC_LABELS)
    topic_table['label'] = topic_table['label'].fillna(topic_table['frex_label'])
    
    # ── HYPOTHESIS CLASSIFICATION ──────────────────────────────────────────
    # ⚠️ ACTION REQUIRED: Fill this in based on your FREX terms + direction
    TOPIC_HYPOTHESIS = {
        1:  None,   # Freelance AI training work
        2:  None,   # Survey & market research
        3:  None,   # NLP & text annotation
        4:  None,   # B2B outsourcing content
        5:  None,   # Technical AI evaluation tasks
        6:  None,   # AI governance & oversight
        7:  None,   # LLM evaluation & fine-tuning
        8:  None,   # Cookie & tracking notices
        9:  None,   # Computer vision labeling
        10: None,   # Privacy policy & terms of service
        11: None,   # Worker payments & earnings
        12: None,   # Annotation platform tools
        13: None,   # Government & compliance hiring
        14: None,   # Expert specialist recruitment
        15: None,   # Enterprise partnerships
        16: None,   # B2B workforce solutions
        17: None,   # Annotation API & developer tools
        18: None,   # Human-AI discourse
        19: None,   # Speech & audio data collection
        20: None,   # Autonomous vehicles & drone sensing
        21: None,   # AI deployment & synthetic data
        22: None,   # Microwork platform campaigns
        23: None,   # Medical image annotation
        24: None,   # Search relevance & content tagging
        25: None,   # Machine learning concepts
    }
    topic_table['hypothesis'] = topic_table['topic_id'].map(TOPIC_HYPOTHESIS)
    
    print('=== Full Topic Table (sorted by estimate) ===')
    display_cols = ['topic_id', 'label', 'frex_terms', 'estimate',
                    'ci_lower', 'ci_upper', 'significant', 'hypothesis']
    print(topic_table[display_cols].to_string(index=False))
else:
    print('STM tables not available.')

In [ ]:
# ── §4.3.2: STM Prevalence Forest Plot (Figure 4.6) ───────────────────────
# All 25 topics sorted by estimate, with 95% CI, coloured by direction

if 'stm_prevalence' in tables:
    prev = pd.read_sql("""
        SELECT topic_id, estimate, ci_lower, ci_upper, significant
        FROM stm_prevalence ORDER BY estimate
    """, conn)
    
    prev['label'] = prev['topic_id'].map(TOPIC_LABELS)
    prev['hyp'] = prev['topic_id'].map(TOPIC_HYPOTHESIS)
    
    fig, ax = plt.subplots(figsize=(10, 12))
    
    for i, (_, row) in enumerate(prev.iterrows()):
        color = PAL_HYP.get(row['hyp'], C_SHARED)
        alpha = 1.0 if row['significant'] else 0.4
        ax.plot([row['ci_lower'], row['ci_upper']], [i, i],
                color=color, linewidth=2, alpha=alpha)
        ax.plot(row['estimate'], i, 'o', color=color,
                markersize=6, alpha=alpha)
    
    ax.axvline(0, color=C_TEXT, linewidth=0.8, linestyle='-', alpha=0.5)
    ax.set_yticks(range(len(prev)))
    ax.set_yticklabels(prev['label'], fontsize=8)
    ax.set_xlabel('Prevalence Effect (← worker-prevalent | client-prevalent →)',
                  fontsize=10)
    ax.set_title('STM Topic Prevalence by Audience Type', fontsize=12)
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor=C_WORKER, label='H1a (labour)'),
        mpatches.Patch(facecolor=C_CLIENT, label='H1b (automation)'),
        mpatches.Patch(facecolor=C_H1C, label='H1c (quality signal)'),
        mpatches.Patch(facecolor=C_SHARED, label='Unclassified / shared'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=8)
    
    apply_style(ax)
    save_fig(fig, 'fig_46_stm_prevalence_forest')
    plt.show()

In [ ]:
# ── §4.3.3: Topic Co-occurrence Network (Figure 4.7) ──────────────────────
# NEW FIGURE — computes topic-topic correlations from stm_theta

if 'stm_theta' in tables and _HAS_NX:
    # Build document-topic matrix
    theta_df = pd.read_sql("""
        SELECT page_id, topic_id, theta
        FROM stm_theta
    """, conn)
    
    theta_wide = theta_df.pivot(index='page_id', columns='topic_id', values='theta')
    
    # Compute topic-topic correlation matrix
    corr = theta_wide.corr()
    
    # Build network: edges where correlation > threshold
    CORR_THRESHOLD = 0.05
    
    G = nx.Graph()
    for tid in corr.columns:
        G.add_node(tid)
    
    for i in corr.columns:
        for j in corr.columns:
            if i < j and corr.loc[i, j] > CORR_THRESHOLD:
                G.add_edge(i, j, weight=corr.loc[i, j])
    
    # Node attributes
    mean_prev = theta_wide.mean()
    
    # Layout
    pos = nx.spring_layout(G, k=2.5, seed=42, weight='weight')
    
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Draw edges
    for (u, v, d) in G.edges(data=True):
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]],
                color=C_GRID, linewidth=d['weight'] * 20, alpha=0.5)
    
    # Draw nodes
    for tid in G.nodes:
        hyp = TOPIC_HYPOTHESIS.get(tid)
        color = PAL_HYP.get(hyp, C_SHARED)
        size = mean_prev.get(tid, 0.04) * 3000
        ax.scatter(pos[tid][0], pos[tid][1], s=size, c=color,
                   edgecolors='white', linewidth=1, zorder=3)
        label = TOPIC_LABELS.get(tid, f'T{tid}')
        # Shorten label
        short = label[:25] + '...' if len(label) > 28 else label
        ax.annotate(short, pos[tid], fontsize=7, ha='center', va='bottom',
                    xytext=(0, 8), textcoords='offset points')
    
    ax.set_title('Topic Co-occurrence Network', fontsize=13)
    ax.axis('off')
    
    legend_elements = [
        mpatches.Patch(facecolor=C_WORKER, label='H1a (labour)'),
        mpatches.Patch(facecolor=C_CLIENT, label='H1b (automation)'),
        mpatches.Patch(facecolor=C_H1C, label='H1c (quality signal)'),
        mpatches.Patch(facecolor=C_SHARED, label='Shared / unclassified'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
    
    save_fig(fig, 'fig_47_topic_cooccurrence_network')
    plt.show()
    
    print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges '
          f'(threshold ρ > {CORR_THRESHOLD})')
elif not _HAS_NX:
    print('SKIP: networkx not installed. Run: pip install networkx')
else:
    print('SKIP: stm_theta not in DB.')

---
## §4.4 — Interpretive Synthesis: H1a–H1c with Close Reading

This section does not produce new figures but **selects and re-presents** evidence from §4.2–4.3, supplemented by close reading passages.  
The cells below extract the specific numbers and passages needed.

In [ ]:
# ── §4.4.1: H1a Evidence Summary ──────────────────────────────────────────
# Pull together: labour keyness terms, exclusivity stats, STM labour topics

print('=== H1a: Labour Visibility Gap ===')
print()

# Key labour terms in worker register
labour_terms = pd.read_sql("""
    SELECT term, g2, freq_worker_pmw, freq_client_pmw, exclusivity
    FROM keyness_results
    WHERE direction = 'worker'
      AND term IN ('worker', 'task', 'earn', 'annotator', 'qualify',
                   'apply', 'payment', 'job', 'project', 'gig')
    ORDER BY g2 DESC
""", conn)
print('Key labour terms (worker register):')
print(labour_terms.to_string(index=False))

# How many labour terms have E < -0.5?
if 'term_exclusivity' in tables:
    n_excl = conn.execute("""
        SELECT COUNT(*) FROM term_exclusivity
        WHERE exclusivity_index < -0.5
    """).fetchone()[0]
    print(f'\nTerms with E < -0.5 (worker-exclusive): {n_excl}')

# STM labour topics
if 'stm_prevalence' in tables:
    h1a_topics = [tid for tid, h in TOPIC_HYPOTHESIS.items() if h == 'H1a']
    if h1a_topics:
        placeholders = ','.join('?' * len(h1a_topics))
        h1a_prev = pd.read_sql(f"""
            SELECT topic_id, estimate, ci_lower, ci_upper, significant
            FROM stm_prevalence
            WHERE topic_id IN ({placeholders})
            ORDER BY estimate
        """, conn, params=h1a_topics)
        h1a_prev['label'] = h1a_prev['topic_id'].map(TOPIC_LABELS)
        print('\nH1a-classified STM topics:')
        print(h1a_prev.to_string(index=False))
    else:
        print('\n⚠️ No topics classified as H1a yet — fill TOPIC_HYPOTHESIS above.')

In [ ]:
# ── §4.4.2: H1b Evidence Summary ──────────────────────────────────────────

print('=== H1b: Automation Narrative ===')
print()

# Client keyness: automation terms
auto_terms = pd.read_sql("""
    SELECT term, g2, freq_client_pmw, freq_worker_pmw, exclusivity
    FROM keyness_results
    WHERE direction = 'client'
      AND term IN ('model', 'solution', 'ai', 'pipeline', 'deploy',
                   'automate', 'algorithm', 'intelligent', 'autonomous')
    ORDER BY g2 DESC
""", conn)
print('Key automation terms (client register):')
print(auto_terms.to_string(index=False))

# Co-occurrence decoupling: top collocates of 'automate' in client register
auto_cooc = get_pmi_profile('automate', 'client', 10)
if len(auto_cooc) > 0:
    print(f'\nTop collocates of "automate" in client register:')
    print(auto_cooc.to_string(index=False))
    # Check if any are labour terms
    labour_set = {'worker', 'annotator', 'task', 'earn', 'human', 'labour', 'job'}
    overlap = set(auto_cooc['collocate']) & labour_set
    print(f'Labour terms among top collocates: {overlap if overlap else "NONE"}')

In [ ]:
# ── §4.4.3: H1c Evidence Summary ──────────────────────────────────────────

print('=== H1c: Strategic Hypervisibility ===')
print()

# "human" keyness in client register
human_key = pd.read_sql("""
    SELECT term, g2, freq_client_pmw, freq_worker_pmw, exclusivity
    FROM keyness_results
    WHERE term = 'human'
""", conn)
print('Keyness of "human":')
print(human_key.to_string(index=False))

# "human_in_the_loop" if it exists
hitl = pd.read_sql("""
    SELECT term, g2, freq_client_pmw, freq_worker_pmw, exclusivity
    FROM keyness_results
    WHERE term LIKE '%human%loop%'
""", conn)
if len(hitl) > 0:
    print('\n"human-in-the-loop" compound:')
    print(hitl.to_string(index=False))

# Recap: human co-occurrence divergence (already shown in §4.2.3)
print('\n(See Figure 4.4 for "human" PMI comparison)')

# H1c STM topic
h1c_topics = [tid for tid, h in TOPIC_HYPOTHESIS.items() if h == 'H1c']
if h1c_topics and 'stm_prevalence' in tables:
    placeholders = ','.join('?' * len(h1c_topics))
    h1c_prev = pd.read_sql(f"""
        SELECT topic_id, estimate, ci_lower, ci_upper, significant
        FROM stm_prevalence
        WHERE topic_id IN ({placeholders})
    """, conn, params=h1c_topics)
    h1c_prev['label'] = h1c_prev['topic_id'].map(TOPIC_LABELS)
    print('\nH1c-classified STM topics:')
    print(h1c_prev.to_string(index=False))
else:
    print('\n⚠️ No topics classified as H1c yet.')

In [ ]:
# ── §4.4: KWIC / Close Reading Passages ───────────────────────────────────
# Retrieve highest-θ documents for hypothesis-classified topics
# for use in close reading

KWIC_N = 3  # passages per topic

for hyp_label in ['H1a', 'H1b', 'H1c']:
    topics = [tid for tid, h in TOPIC_HYPOTHESIS.items() if h == hyp_label]
    if not topics:
        print(f'\n{hyp_label}: no topics classified — skipping')
        continue
    
    print(f'\n{"="*60}')
    print(f'{hyp_label} — Top passages from highest-θ documents')
    print(f'{"="*60}')
    
    for tid in topics:
        label = TOPIC_LABELS.get(tid, f'Topic {tid}')
        print(f'\n--- Topic {tid}: {label} ---')
        
        # Get top documents by theta for this topic
        if 'stm_topic_profile' in tables:
            docs = conn.execute("""
                SELECT p.page_id, p.audience, p.domain, p.theta
                FROM stm_topic_profile p
                WHERE p.topic_id = ?
                ORDER BY p.theta DESC
                LIMIT ?
            """, (tid, KWIC_N)).fetchall()
        elif 'stm_theta' in tables:
            docs = conn.execute("""
                SELECT t.page_id, c.audience, c.domain, t.theta
                FROM stm_theta t
                JOIN corpus_view c ON t.page_id = c.page_id
                WHERE t.topic_id = ?
                ORDER BY t.theta DESC
                LIMIT ?
            """, (tid, KWIC_N)).fetchall()
        else:
            docs = []
        
        for doc in docs:
            page_id, audience, domain, theta = doc
            # Get first 500 chars of text
            text = conn.execute("""
                SELECT SUBSTR(text_content, 1, 500) FROM pages
                WHERE page_id = ?
            """, (page_id,)).fetchone()
            snippet = text[0] if text else '[no text]'
            print(f'  [{audience}] {domain} (θ={theta:.3f})')
            print(f'  {snippet[:300]}...')
            print()

In [ ]:
# ── Cleanup ────────────────────────────────────────────────────────────────
conn.close()
print('Database connection closed.')
print(f'\nAll figures saved to: {FIG_DIR}/')
print('\nFigure inventory:')
for f in sorted(FIG_DIR.glob('*')):
    print(f'  {f.name}')